In [0]:
%sql
CREATE OR REPLACE TABLE crm_ingestion.gold.health_metrics AS (
    SELECT
        b.lead_id,
        b.status_label,
        b.date_updated,
        -- Real client name comes from lead_dim itself now (3_04 v5, Nov 11 2026):
        -- LEAD_NAME holds real names (1,020/1,381 CC leads) with hash fallbacks.
        -- The inline Calendly "Invitee:" regex join (843/1,380 coverage) was removed
        -- Nov 12 — superseded and out-covered by 3_04.
        COALESCE(b.lead_name, b.lead_user_hash_name, b.lead_display_name) AS lead_name,
        b.lead_user_hash_name,
        b.lead_display_name,
        b.customer_emails,
        b.primary_customer_email,
        b.csm_id,
        csm.csm_name,
        csm.csm_email,
        b.Contracted_Value,
        b.Tier,
        b.Avatar,
        sd.date_of_sale,
        cs.Contract_Start_Date AS contract_start_date,
        sd.program,
        sd.setter_name,
        sd.closer_name,
        sd.contract_value AS sale_contract_value,
        e.days_since_last_email,
        m.days_since_last_meeting,
        COALESCE(mc.missed_calls_14_days, 0) AS missed_calls_14_days,
        nm.days_until_next_meeting,
        l.days_since_last_login,
        l.last_login_date,
        l.last_login_display,
        cl.days_since_last_meeting_calendly,
        cl.last_booking_date,
        s.sentiment,
        s.days_since_last_message_from_client,
        s.days_since_last_message_from_team_member,
        (sh.staff_hash IS NOT NULL) AS is_intern_client,
        (spine.h IS NULL) AS identity_unlinked,
        (b.customer_emails IS NULL OR size(b.customer_emails) = 0) AS no_client_email
    FROM crm_ingestion.gold.lead_dim b
    -- Email columns come straight from lead_dim (3_04 v4 absorbed lead_emails);
    -- gold.lead_emails is retired/stale (218 leads missing from it) — not joined.
    LEFT JOIN crm_ingestion.gold.sales_fact sd ON b.lead_id = sd.lead_id
    -- Contract start from the CRM's stored lead custom field (latest snapshot per
    -- lead). Matches the reference output exactly (verified 3/3 on traced leads);
    -- covers 1,334 of 1,371 clients (37 residual = no custom field and no
    -- qualifying sale activity — verified source gap, not a pipeline issue). The sale-activity date stays available
    -- separately as date_of_sale — they disagree 141/172 where both exist.
    LEFT JOIN (
        SELECT leads_processed_id AS lead_id, Contract_Start_Date
        FROM crm_ingestion.silver.leads_processed_custom
        QUALIFY ROW_NUMBER() OVER (PARTITION BY leads_processed_id
                                   ORDER BY bronze_insert_date DESC) = 1
    ) cs ON cs.lead_id = b.lead_id
    LEFT JOIN crm_ingestion.gold.staff_dim csm ON b.csm_id = csm.csm_id
    LEFT JOIN crm_ingestion.gold.days_since_last_email e ON b.lead_id = e.lead_id
    LEFT JOIN crm_ingestion.gold.days_since_last_meeting m ON b.lead_id = m.lead_id
    LEFT JOIN crm_ingestion.gold.missed_calls_14_days mc ON b.lead_id = mc.lead_id
    LEFT JOIN crm_ingestion.gold.days_until_next_meeting nm ON b.lead_id = nm.lead_id
    LEFT JOIN crm_ingestion.gold.days_since_last_login l ON b.lead_id = l.lead_id
    -- 3_10 re-keyed per invitee email (spec, Nov 12) — join back through the
    -- 2_8 IDENTITY SPINE's email closure (real_emails → lead_ids), NOT the lead
    -- record's own customer_emails: the spine closure covers the invitee's full
    -- identity cluster. Verified Nov 12: reproduces the old Nov 10 spine-bridged
    -- values EXACTLY (1,132/1,381 CC leads covered, 0 value changes, 0 lost) —
    -- while a lead_dim.customer_emails join loses 52 leads and stales 311 values
    -- (bookings made under identity-cluster emails not on the lead record).
    LEFT JOIN (
        SELECT s.lead_id,
               MIN(cl.days_since_last_meeting_calendly) AS days_since_last_meeting_calendly,
               MIN(cl.last_booking_date) AS last_booking_date
        FROM (
            SELECT lead_id, email
            FROM (
                SELECT explode(lead_ids) AS lead_id, real_emails
                FROM crm_ingestion.silver.user_identity_map
            )
            LATERAL VIEW explode(real_emails) re AS email
        ) s
        JOIN crm_ingestion.gold.days_since_last_meeting_calendly cl
          ON cl.invitee_email = s.email
        GROUP BY s.lead_id
    ) cl ON cl.lead_id = b.lead_id
    LEFT JOIN crm_ingestion.gold.sentiment s ON b.lead_id = s.lead_id
    -- INTERN CLIENT: the lead's own identity hash belongs to a staff person —
    -- a former client who now works here. The client-to-coach conversion is
    -- one-way: once staff, never a regular client again. They stay in this
    -- table, flagged, for tracking; 3_14 excludes them from the client book.
    LEFT JOIN (
        SELECT DISTINCT explode(user_32digits) AS staff_hash
        FROM crm_ingestion.gold.staff_emails
    ) sh ON sh.staff_hash = b.lead_user_hash_name  -- identity hash — LEAD_NAME is now a REAL name, would never match
    -- ATTENTION FLAGS — issues that need eyes, each on its own column:
    --   identity_unlinked: the lead's hash exists nowhere in the identity
    --     spine — client or staff cannot be proven. Includes the 11
    --     staff-contact-card leads ($60K contracts) that might be intern
    --     clients in disguise. Needs human review.
    --   no_client_email: every email ever attached to this lead is
    --     staff-owned (or none exists) — unreachable by email today.
    LEFT JOIN (
        SELECT DISTINCT explode(user_32digits) AS h
        FROM crm_ingestion.silver.user_identity_map
    ) spine ON spine.h = b.lead_user_hash_name  -- identity hash — LEAD_NAME is now a REAL name, would never match
    -- Population: coaching clients only. lead_dim became the ALL-LEADS spine
    -- (66,782) in 3_04 v4/v5; without this filter the assembly would emit every
    -- lead in the CRM (48x the reference population). Nov 10 build = 1,381 rows.
    WHERE b.status_label = 'Coaching Client'
)